# Regrid reanalyses to 1-degree grid

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import xagg as xa
import xesmf as xe
import os
import glob
import re

from funcs_support import get_filepaths,get_params,utility_save
dir_list = get_params()

In [2]:
from distributed import Client
client = Client()
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/proxy/8787/status,Workers: 6
Total threads: 18,Total memory: 72.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42947,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:41899,Total threads: 3
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/proxy/39185/status,Memory: 12.00 GiB
Nanny: tcp://127.0.0.1:45489,


In [3]:
df = get_filepaths()

In [4]:
params_subset = {'lat':slice(-56,86),'lon':slice(-180,180)}
# Get reference 1x1 grid
ref_grid = xr.Dataset(coords = {'lat':(['lat'],np.arange(params_subset['lat'].start+0.5,
														 params_subset['lat'].stop-0.4)),
								'lon':(['lon'],np.arange(params_subset['lon'].start+0.5,
														 params_subset['lon'].stop-0.4))})


In [5]:
modlist = ['UDel','ERA5-025','MERRA2','JRA-3Q','GMFD','PRISM']
varlist = ['tas','tasmax','pr','prcorr']

In [6]:
df = df.query('run == "reanalysis" or run == "obs"')
df = df.iloc[[mod in modlist for mod in df.model.values],:]
df = df.iloc[[var in varlist for var in df.varname.values],:]

In [11]:
for row in df.iterrows():
    output_fn = re.sub(row[1]['model'],row[1]['model']+'-1deg',row[1]['path'])
    
    if not os.path.exists(output_fn):
        if row[1]['filetype'] == 'zarr':
            ds = xr.open_zarr(row[1]['path'])
        else:
            ds = xr.open_dataset(row[1]['path'])
        if 'pr' in row[1]['varname']:
            rgrd_typ = 'conservative'
        else:
            rgrd_typ = 'bilinear'

        if (((ds.lon.max() > 175) and (ds.lon.min() < -175)) or 
            ((ds.lon.max() > 355) and (ds.lon.min() < 5))):
            periodic = True
        else:
            periodic = False
        rgrd = xe.Regridder(ds,ref_grid,rgrd_typ,ignore_degenerate=True,periodic=periodic)
        ds = rgrd(ds)
        
        ds.attrs['SOURCE'] = '0a_create_1deg_reanalyses.ipynb'
        ds.attrs['DESCRIPTION'] = f'Data, regridded using {rgrd_typ} algorithm to 1-degree BCD-ME grid from {re.split(r'\/',row[1]['path'])[-1]}'
        if row[1]['filetype'] == 'zarr':
            ds = ds.drop_encoding()
        utility_save(ds,output_fn)
    else:
        print(output_fn+' already processed, skipped.')
# ~ 30 minutes for all but PRISM on 18 cores, about 1.5 minutes extra for PRISM

/glade/derecho/scratch/schwarzwald/climate_raw_derecho/PRISM-1deg/tasmax_day_PRISM-1deg_historical_obs_19810101-20111231.zarr already processed, skipped.
/glade/derecho/scratch/schwarzwald/climate_raw_derecho/PRISM-1deg/tas_day_PRISM-1deg_historical_obs_19860101-20111231.zarr already processed, skipped.
/glade/derecho/scratch/schwarzwald/climate_raw_derecho/MERRA2-1deg/prcorr_Amon_MERRA2-1deg_historical_reanalysis_19800101-20250530.nc already processed, skipped.
/glade/derecho/scratch/schwarzwald/climate_raw_derecho/MERRA2-1deg/tasmax_day_MERRA2-1deg_historical_reanalysis_19800101-20241231.nc already processed, skipped.
/glade/derecho/scratch/schwarzwald/climate_raw_derecho/MERRA2-1deg/pr_day_MERRA2-1deg_historical_reanalysis_19800101-20231231.nc already processed, skipped.
/glade/derecho/scratch/schwarzwald/climate_raw_derecho/MERRA2-1deg/pr_Amon_MERRA2-1deg_historical_reanalysis_19800101-20250530.nc already processed, skipped.
/glade/derecho/scratch/schwarzwald/climate_raw_derecho/ME